---
title: "HMM"
subtitle: "Hidden Markov Models in Pyro"
description: "Hidden Markov Model with Poisson emissions"
author: ["Tommaso Piscitelli", "Erik De Luca"]
categories: ["Pyro", "Python", "HMM"]
date: 2025-05-24
---

Load dataset

In [ ]:
# !pip install pyprojroot


In [ ]:
import pandas as pd
import numpy as np
from hmmlearn import hmm
import matplotlib.pyplot as plt
from pyprojroot import here
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv(here("data/recent_donations.csv"))
df

In [ ]:


df["gender"] = df["gender"].map({"M": 0, "F": 1})


scaler = MinMaxScaler()
df[["birth_year", "first_donation_year"]] = scaler.fit_transform(df[["birth_year", "first_donation_year"]])

year_cols = sorted([col for col in df.columns if col.startswith("y_")])
donation_data = df[year_cols].values  # matrice [n_donatori, T]

features = df[["gender", "birth_year", "first_donation_year"]].values  # shape: [n_donatori, n_features]


Let's start with a simple Hidden Markov Model. In this case we assume that the emmissions are continuous and not discrete

In [ ]:

# Estrai le colonne delle osservazioni
y_cols = [col for col in df.columns if col.startswith("y_")]
sequences = df[y_cols].to_numpy()

# Prepara i dati per hmmlearn
X = sequences.reshape(-1, 1)  # hmmlearn richiede forma (n_samples, n_features)
lengths = [len(y_cols)] * sequences.shape[0]  # una sequenza per ogni individuo

# Costruisci l'HMM (es: GaussianHMM, Poisson non è supportato direttamente)
model = hmm.GaussianHMM(n_components=3, covariance_type="diag", n_iter=100, random_state=42)
model.fit(X, lengths)


In [ ]:
states = model.predict(X)  # X = osservazioni (n_donors * T, 1)

plt.figure(figsize=(15, 4))
plt.plot(X, label="Donazioni")
plt.plot(states, label="Stato latente", alpha=0.7)
plt.legend()
plt.title("Donazioni e stati latenti (HMM Gaussian)")
plt.show()


In [ ]:
import seaborn as sns

for i in range(model.n_components):
    mu = model.means_[i][0]
    sigma = np.sqrt(model.covars_[i][0])
    sns.kdeplot(np.random.normal(mu, sigma, 1000), label=f"Stato {i}")

plt.title("Distribuzioni delle emissioni")
plt.legend()
plt.show()


## Pyro

In [ ]:
import torch
# Estrai la matrice y: righe = individui, colonne = anni
donation_cols = [col for col in df.columns if col.startswith("y_")]
Y = df[donation_cols].fillna(0).astype(int).to_numpy()  # [n_donors, T]
Y_tensor = torch.tensor(Y, dtype=torch.float32)  # Pyro lavora con torch

In [ ]:
import pyro
import pyro.distributions as dist
from pyro.nn import PyroParam
from pyro.infer import SVI, TraceEnum_ELBO, config_enumerate
from pyro.optim import Adam
from torch.distributions import constraints
from pyro.nn import PyroModule


class SimpleDiscreteHMM:
    def __init__(self, num_states=3):
        self.num_states = num_states
        self.lambdas = PyroParam(torch.ones(num_states), constraint=constraints.positive)
        self.start_probs = PyroParam(torch.ones(num_states) / num_states, constraint=constraints.simplex)
        self.trans_probs = PyroParam(torch.ones(num_states, num_states) / num_states, constraint=constraints.simplex)

    @config_enumerate
    def model(self, Y):
        n_donors, T = Y.shape

        with pyro.plate("donors", n_donors, dim=-2):
            z_prev = pyro.sample("z_0", dist.Categorical(pyro.param("start_probs")))

            for t in range(T):
                z_t = pyro.sample(f"z_{t}", dist.Categorical(pyro.param("trans_probs")[z_prev]))
                pyro.sample(f"y_{t}", dist.Poisson(self.lambdas[z_t]), obs=Y[:, t])
                z_prev = z_t


class Guide(PyroModule):
    def __init__(self, num_states):
        super().__init__()
        self.num_states = num_states

    @config_enumerate
    def forward(self, Y):
        pass  # nessun parametro latente globale da approssimare qui

In [ ]:
model = SimpleDiscreteHMM(num_states=3)
guide = Guide(num_states=3)

optimizer = Adam({"lr": 0.01})
svi = SVI(model.model, guide, optimizer, loss=TraceEnum_ELBO())

for step in range(500):
    loss = svi.step(Y_tensor)
    if step % 50 == 0:
        print(f"Step {step}: loss = {loss:.2f}")



In [ ]:
print("Lambda per stato:", model.lambdas.detach().numpy())
print("Probabilità iniziali:", model.start_probs.detach().numpy())
print("Matrice di transizione:", model.trans_probs.detach().numpy())
